# Siamese

## 0. Imports

In [2]:
import os
import sys
from dataclasses import dataclass

# Get the absolute path of the parent directory
parent_dir = os.path.abspath(os.path.join(os.getcwd(), ".."))

# Add the parent directory to sys.path if it's not already there
if parent_dir not in sys.path:
    sys.path.append(parent_dir)

In [3]:
import pandas as pd
from pathlib import Path
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import random
from torch.utils.data import Dataset, DataLoader
import torch
import torch.nn as nn
import itertools
import evaluation as evals


## 1. Load Data

In [4]:
data_dir = Path("../../data/")

# read features
features_path = data_dir / "features_df.csv"
features_df = pd.read_csv(f"{features_path}")
print(f"Features Shape: {features_df.shape}")

Features Shape: (1932, 110)


In [5]:
# optionally read the edgelist in for now
edgelist_path = data_dir / "edgelists" / "edgelist_full.csv"
edgelist = pd.read_csv(f"{edgelist_path}")
print(f"Edgelist Shape: {edgelist.shape}")

Edgelist Shape: (3730692, 3)


## 2. Pre-processing

In [6]:
# save user ids for mapping
user_ids = features_df["user_id"].values

# one-hot encode all categorical
cat_cols = features_df.select_dtypes(include=["object", "bool"]).columns.tolist()
if "user_id" in cat_cols:
    cat_cols.remove("user_id")
df_numeric = features_df.drop(columns=["user_id"])
df_numeric = pd.get_dummies(df_numeric, columns=cat_cols)
print("One-hot encoding complete")

# fillna with median
df_numeric = df_numeric.fillna(df_numeric.median())
print("Filled missing with median")

# standardize and construct the X matrix
scaler = StandardScaler()
X = scaler.fit_transform(df_numeric)
print(X.shape)

X_train, X_test, ids_train, ids_test = train_test_split(
    X, user_ids, test_size=0.2, random_state=42
)

print(f"Train Shape: {X_train.shape}, Test Shape: {X_test.shape}")

One-hot encoding complete
Filled missing with median
(1932, 116)
Train Shape: (1545, 116), Test Shape: (387, 116)


/var/folders/jh/9_qy7nd96v9_q0_x1sffyq9c0000gn/T/ipykernel_188/2416148417.py:5: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = features_df.select_dtypes(include=["object", "bool"]).columns.tolist()


## 3. Training Siamese

Here's where the fun stuff starts

### Why is it complex?

The structure of our data for training is kind of like this matrix:

$$
\begin{bmatrix}
 \text{Features(Anchor User)} \\
 \text{Features(Positive Match)} \\
 \text{Features(Negative Match)} \\
\end{bmatrix}
$$

We have a set of all these matrices, across all our users. This isn't a typical csv file 
that we can just pass in as an input. So, we need to define a custom `Dataset` class – 
TripletDataset. This tells torch how to read our data during training. We connect 
TripletDataset dataset class to the edge list (containing labels) and the features_df 
containing all the features. 

In [7]:
class TripletDataset(Dataset):
    def __init__(self, edge_list, features_df):
        self.edge_list = edge_list.reset_index(drop=True)
        self.features_df = features_df

        # keep a list of all users for easy sampling
        self.all_users = list(features_df.index)

    def __len__(self):
        return len(self.edge_list)

    def __getitem__(self, idx):
        # 1. Look up the positive pair (match) from the edge list
        row = self.edge_list.iloc[idx]
        anchor_id = row["user_anchor"]
        positive_id = row["user_match"]

        # 2. Sample a random Negative user
        negative_id = random.choice(self.all_users)
        while negative_id == anchor_id or negative_id == positive_id:
            negative_id = random.choice(
                self.all_users
            )  # Keep trying until we get a non-match

        # 3. Fetch their feature vectors and convert to PyTorch Tensors
        # (We use float32 because neural network weights default to float32)
        feat_a = torch.tensor(
            self.features_df.loc[anchor_id].values, dtype=torch.float32
        )
        feat_p = torch.tensor(
            self.features_df.loc[positive_id].values, dtype=torch.float32
        )
        feat_n = torch.tensor(
            self.features_df.loc[negative_id].values, dtype=torch.float32
        )

        return feat_a, feat_p, feat_n

In [8]:
# use gpu when available
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print(f"currently using device: {device}")

currently using device: mps


### Model

Simple neural network. Used basic architecture settings from Philip's class

In [9]:
class UserEmbeddingMLP(nn.Module):
    def __init__(self, input_dim, embedding_dim, layers, dropout_rate):
        super(UserEmbeddingMLP, self).__init__()

        network_layers = []
        prev_dim = input_dim

        for layer_size in layers:
            network_layers.append(nn.Linear(prev_dim, layer_size))
            network_layers.append(nn.ReLU())
            network_layers.append(nn.Dropout(dropout_rate))
            prev_dim = layer_size

        network_layers.append(nn.Linear(prev_dim, embedding_dim))
        self.network = nn.Sequential(*network_layers)

    def forward(self, x):
        return self.network(x)

In [10]:
@dataclass
class Config:
    embedding_dim: int
    layers: list
    learning_rate: float
    dropout_rate: float
    batch_size: int
    margin: float
    epochs: int = 10

    def __str__(self) -> str:
        parts = [
            f"model_emb{self.embedding_dim}",
            f"dep{len(self.layers)}",
            f"lr{self.learning_rate}",
            f"drop{self.dropout_rate}",
            f"batch{self.batch_size}",
            f"margin{self.margin}",
        ]
        return "_".join(parts)


grid = {
    "embedding_dim": [16, 32, 64],
    "layers": [[128, 64], [256, 128, 64]],
    "learning_rate": [0.001, 0.0025],
    "dropout_rate": [0.1, 0.2],
    "batch_size": [64, 128],
    "margin": [0.5, 1.0, 1.5],
}

keys = grid.keys()
values = grid.values()
all_configs = [Config(**dict(zip(keys, v))) for v in itertools.product(*values)]

print("Total configs:", len(all_configs))

Total configs: 144


### Training Loop

configs are right on top

In [11]:
def train_siamese(config, edge_list, features_df):
    INPUT_DIM = features_df.shape[1]

    dataset = TripletDataset(edge_list, features_df)
    dataloader = DataLoader(dataset, batch_size=config.batch_size, shuffle=True)

    model = UserEmbeddingMLP(
        input_dim=INPUT_DIM,
        embedding_dim=config.embedding_dim,
        layers=config.layers,
        dropout_rate=config.dropout_rate,
    ).to(device)

    criterion = nn.TripletMarginLoss(margin=config.margin, p=2)
    optimizer = torch.optim.Adam(model.parameters(), lr=config.learning_rate)

    model.train()
    loss_history = []

    for epoch in range(config.epochs):
        epoch_loss = 0.0

        for batch_a, batch_p, batch_n in dataloader:
            optimizer.zero_grad()

            emb_a = model(batch_a.to(device))
            emb_p = model(batch_p.to(device))
            emb_n = model(batch_n.to(device))

            loss = criterion(emb_a, emb_p, emb_n)
            loss.backward()
            optimizer.step()

            epoch_loss += loss.item()

        avg_epoch_loss = epoch_loss / len(dataloader)
        loss_history.append(avg_epoch_loss)

    return model, loss_history[-1]

In [12]:
# new grid search
from datetime import datetime

run_id = datetime.now().strftime("%Y%m%d_%H%M")
save_dir = f"experiments/siamese_{run_id}"
os.makedirs(save_dir, exist_ok=True)

results = []

for config in all_configs:
    # """
    # tested flow with a subset of the hyperparameters
    # """
    # test_configs = all_configs[:2]
    # for config in test_configs:
    # try with a small number of epochs
    config.epochs = 2
    print("Training config:", config)

    """
    added X_train_df because cannot find scaled_feature_df used in old code
    also there's no user_id_anchor or user_id_positive columns so used user_anchor and user_match instead   
    """
    X_train_df = pd.DataFrame(X_train, index=ids_train)
    train_users_set = set(ids_train)
    train_edgelist = edgelist[
        edgelist["user_anchor"].isin(train_users_set)
        & edgelist["user_match"].isin(train_users_set)
    ].copy()

    """
    ran on subset of edgelist cuz it's taking too long due to the large size of the model
    """
    train_edgelist_small = train_edgelist.head(10000).copy()
    model, final_loss = train_siamese(config, train_edgelist_small, X_train_df)
    # model, final_loss = train_siamese(config, train_edgelist, X_train_df)
    # model, final_loss = train_siamese(config, edgelist, scaled_features_df)

    model_name = f"siamese_{str(config)}"
    model_path = f"{save_dir}/{model_name}.pt"
    torch.save(model.state_dict(), model_path)

    # Generate test set's Embeddings for Evaluation
    model.to(device)
    model.eval()
    with torch.no_grad():
        X_test_tensor = torch.tensor(X_test, dtype=torch.float32).to(device)
        embeddings = model(X_test_tensor).cpu().numpy()

    # evaluator
    evaluator = evals.RecommenderEvaluator(embeddings, ids_test, edgelist)
    metrics = evaluator.get_all_metrics()

    # Flatten config into a dict for the DataFrame
    res_dict = {
        "name": model_name,
        "embedding_dim": config.embedding_dim,
        "depth": len(config.layers),
        "lr": config.learning_rate,
        "dropout_rate": config.dropout_rate,
        "batch_size": config.batch_size,
        "margin": config.margin,
        "final_loss": final_loss,
        "model_path": model_path,
        **metrics,
    }

    results.append(res_dict)
    print(f"Final loss {final_loss}, saved to {model_path}")

# save to CSV
results_df = pd.DataFrame(results).sort_values(by="final_loss")
results_df.to_csv(f"{save_dir}/manifest.csv", index=False)

Training config: model_emb16_dep2_lr0.001_drop0.1_batch64_margin0.5
Final loss 0.506702797617882, saved to experiments/siamese_20260327_1248/siamese_model_emb16_dep2_lr0.001_drop0.1_batch64_margin0.5.pt
Training config: model_emb16_dep2_lr0.001_drop0.1_batch64_margin1.0
Final loss 1.0071311095717606, saved to experiments/siamese_20260327_1248/siamese_model_emb16_dep2_lr0.001_drop0.1_batch64_margin1.0.pt
Training config: model_emb16_dep2_lr0.001_drop0.1_batch64_margin1.5
Final loss 1.5103213787078857, saved to experiments/siamese_20260327_1248/siamese_model_emb16_dep2_lr0.001_drop0.1_batch64_margin1.5.pt
Training config: model_emb16_dep2_lr0.001_drop0.1_batch128_margin0.5
Final loss 0.5055304282828222, saved to experiments/siamese_20260327_1248/siamese_model_emb16_dep2_lr0.001_drop0.1_batch128_margin0.5.pt
Training config: model_emb16_dep2_lr0.001_drop0.1_batch128_margin1.0
Final loss 1.0063776313504087, saved to experiments/siamese_20260327_1248/siamese_model_emb16_dep2_lr0.001_drop0.1